# 05k-d — Architecture failure reassessment

Decisione riproducibile sul solo artefatto development 05k-c. Nessun training, GPU o accesso al fresh test.

## 1. Checkout e input esatto

In [ ]:
import importlib,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches()
from src.hayflow_model.hines_architecture_failure_reassessment import EXPECTED_05KC_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def index_matches(path,expected):
 path=Path(path)
 try:
  if path.is_file():
   with zipfile.ZipFile(path) as archive:
    names=[name for name in archive.namelist() if name.replace('\\','/').endswith('artifact_index.json')];return len(names)==1 and hashlib.sha256(archive.read(names[0])).hexdigest()==expected
  return any(hashlib.sha256(candidate.read_bytes()).hexdigest()==expected for candidate in path.rglob('artifact_index.json'))
 except (OSError,zipfile.BadZipFile):return False
candidates=([Path(os.environ['HAYFLOW_05KC_ARTIFACT']).expanduser()] if os.environ.get('HAYFLOW_05KC_ARTIFACT') else [])+list(INPUT_ROOT.rglob('hayflow_hines_development_autoregressive_repair.zip'))+[p.parent for p in INPUT_ROOT.rglob('development_autoregressive_repair_config.json')]
ARTIFACT_05KC_SOURCE=next((p.resolve() for p in candidates if p.exists() and index_matches(p,EXPECTED_05KC_INDEX_SHA256)),None);assert ARTIFACT_05KC_SOURCE is not None,'Artefatto 05k-c esatto non trovato.'
print({'revision':REVISION,'05k-c':str(ARTIFACT_05KC_SOURCE)})

## 2. Decisione architetturale

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import HinesArchitectureFailureReassessment,HinesArchitectureFailureReassessmentConfig
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_hines_architecture_failure_reassessment.yml').read_text());config=HinesArchitectureFailureReassessmentConfig.from_mapping(cfg['architecture_failure_reassessment'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_hines_architecture_failure_reassessment');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=HinesArchitectureFailureReassessment(OUTPUT_DIR,config,ARTIFACT_05KC_SOURCE,code_revision=REVISION);final_report=session.run();decision=final_report['architecture_reassessment']
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'h2_to_persistence_ratio':decision['h2_to_persistence_rmse_ratio'],'maximum_recommit_improvement':max(decision['recommit_improvement_by_seed'].values()),'maximum_oracle_to_persistence_ratio':max(decision['teacher_reset_oracle_to_persistence_ratio_by_seed'].values()),'current_candidate_retired':final_report['current_candidate_retired'],'authorized_canary':decision['proposed_canary_families'],'full_training':final_report['full_training_authorized'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['full_training_authorized']

## 3. Crea e scarica lo ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_hines_architecture_failure_reassessment','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_kib':round(zip_path.stat().st_size/2**10,2),'download':'avviato dal browser'})